# Collections Analytics — Data Analyst Assignment
## Reconstructing actual recovery performance, data forensics, and the ₹10 Cr recommendation

This notebook is the working analysis behind the deliverables in this repo:
`reports/data_quality_report.md`, `reports/performance_reconstruction.md`,
`reports/statistical_investigation.md`, `reports/counterfactual_methodology.md`,
`reports/investment_recommendation.md`, `reports/production_architecture.md`,
and `reports/executive_memo.docx`.

It runs the SQL pipeline in `sql/00`–`06` against DuckDB and reproduces every
number quoted in those documents, so the reasoning is auditable end to end.

**Structure**
1. Setup
2. Part 1 — Build the Golden dataset (and quantify the cleaning impact)
3. Part 2 — Data forensics (A–G, plus two issues found independently)
4. Question 1 & 3 — Reconstruct performance and test the "11% MoM" claim
5. Part 3 — Statistical investigation of drivers and named biases
6. Part 4 — Counterfactual methodology
7. Question 4 — Where should the ₹10 Cr go
8. Conclusion


## 1. Setup

In [1]:

import duckdb
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)

con = duckdb.connect('../repo_analysis.duckdb')  # fresh local db for reproducibility
import os
os.chdir('..')  # so relative paths in the .sql files (dataset/...) resolve
for f in ['sql/00_load_raw.sql','sql/01_clean_dimensions.sql','sql/02_clean_events.sql',
          'sql/03_golden_build.sql','sql/04_metrics.sql','sql/05_forensics.sql','sql/06_investment_analysis.sql']:
    con.execute(open(f).read())
os.chdir('notebook')
print("Pipeline built OK.")


Pipeline built OK.


In [2]:

tables = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='main' ORDER BY 1").df()
print(len(tables), "tables/views in the working database")
tables.head(60)


54 tables/views in the working database


,table_name
0,clean_account_status_history
1,clean_accounts
2,clean_agents
3,clean_borrowers
4,clean_call_attempts
5,clean_call_dispositions
6,clean_calls
7,clean_campaigns
8,clean_complaints
9,clean_daily_targeting


## 2. Part 1 — Build the Golden dataset

Full documentation of every cleaning decision (source-of-truth choices, entity
resolution, dedup logic, missing-data treatment, timestamp treatment, payment
attribution, historical-change handling, exclusion rules, assumptions) is in
`sql/01_clean_dimensions.sql` and `sql/02_clean_events.sql` as inline comments
next to the code that implements each decision — not repeated here. This
section just shows the **quantified impact**.


In [3]:

dq = con.execute("SELECT * FROM dq_cleaning_impact").df()
dq


,table_name,raw_rows,rows_removed,golden_rows,note
0,borrowers,30600,19585,11015,Deduped to 1 row per borrower_id (latest updat...
1,agents,30000,29000,1000,"Collapsed 30,000 scrambled rows to 1,000 agent..."
2,calls,91350,1350,90000,"Dropped 1,350 duplicate call_id rows (1,271 ex..."
3,whatsapp_events,60600,600,60000,Dropped 600 exact duplicate whatsapp_event_id ...
4,payments,25500,500,25000,Dropped 500 duplicate payment_id rows. Removed...
5,call_dispositions,35000,0,35000,No row-level dedup. Canonicalized PROMISE_TO_P...
6,account_status_history,60000,0,60000,No row-level dedup. 85.7% of accounts disagree...


**Biggest trap avoided:** de-duplicating `payments` by `payment_reference`
(which looks like a natural key) instead of `payment_id` (the actual grain of
one payment event) would have thrown away ~₹17 Cr of real, distinct
part-payments that legitimately share a reference code. See the worked
example below.

In [4]:

# payment_reference is NOT a safe dedup key -- same reference, 3 different real payments
con.execute('''
    SELECT payment_reference, payment_id, event_at, amount, payment_status
    FROM raw_payments WHERE payment_reference = 'TXN0000050468'
    ORDER BY event_at
''').df()


,payment_reference,payment_id,event_at,amount,payment_status
0,TXN0000050468,PAYMENT0012223,2026-02-20 12:53:13,127454.47,SUCCESS
1,TXN0000050468,PAYMENT0012223,2026-02-20 12:53:13,127454.47,SUCCESS
2,TXN0000050468,PAYMENT0010506,2026-06-20 15:55:03,40930.14,SUCCESS
3,TXN0000050468,PAYMENT0010506,2026-06-20 15:55:03,40930.14,SUCCESS
4,TXN0000050468,PAYMENT0006886,2026-07-06 14:04:05,47730.92,SUCCESS


## 3. Part 2 — Data Forensics (A–G, plus two issues found independently)

Full writeup with business impact: `reports/data_quality_report.md`. Each
query below is copied from `sql/05_forensics.sql` so the evidence is visible
inline.

### A. Duplicate payments — CONFIRMED

In [5]:

con.execute("SELECT * FROM raw_payments").df().shape[0], con.execute("SELECT COUNT(DISTINCT payment_id) FROM raw_payments").fetchone()[0]


(25500, 25000)

In [6]:

naive = con.execute("SELECT SUM(amount) FROM raw_payments WHERE payment_status='SUCCESS'").fetchone()[0]
golden = con.execute("SELECT SUM(amount) FROM clean_payments WHERE payment_status='SUCCESS'").fetchone()[0]
print(f"Naive SUCCESS sum:  Rs {naive:,.0f}")
print(f"Golden SUCCESS sum: Rs {golden:,.0f}")
print(f"Overstatement:      Rs {naive-golden:,.0f}  ({100*(naive-golden)/naive:.2f}%)")


Naive SUCCESS sum:  Rs 1,341,485,926
Golden SUCCESS sum: Rs 1,315,583,965
Overstatement:      Rs 25,901,962  (1.93%)


### B. Attribution errors — CONFIRMED (methodology risk)

*(Re-running the exact attribution-window query from `sql/05_forensics.sql` here for transparency.)*

In [7]:

attribution_sql = '''
WITH touches AS (
  SELECT account_id, event_at, campaign_id FROM golden_calls WHERE campaign_id IS NOT NULL
),
pay AS (
  SELECT payment_id, account_id, event_at AS pay_at FROM golden_payments WHERE payment_status='SUCCESS'
),
attr_3d AS (
  SELECT p.payment_id,
         (SELECT t.campaign_id FROM touches t
          WHERE t.account_id = p.account_id AND t.event_at <= p.pay_at
            AND t.event_at >= p.pay_at - INTERVAL 3 DAY
          ORDER BY t.event_at DESC LIMIT 1) AS campaign_3d
  FROM pay p
),
attr_30d AS (
  SELECT p.payment_id,
         (SELECT t.campaign_id FROM touches t
          WHERE t.account_id = p.account_id AND t.event_at <= p.pay_at
            AND t.event_at >= p.pay_at - INTERVAL 30 DAY
          ORDER BY t.event_at DESC LIMIT 1) AS campaign_30d
  FROM pay p
)
SELECT COUNT(*) AS total_success_payments,
       COUNT(a3.campaign_3d)  AS attributable_within_3d,
       COUNT(a30.campaign_30d) AS attributable_within_30d
FROM pay p
JOIN attr_3d a3 USING (payment_id)
JOIN attr_30d a30 USING (payment_id)
'''
con.execute(attribution_sql).df()


,total_success_payments,attributable_within_3d,attributable_within_30d
0,17534,723,5535


### C. Timezone problems — CONFIRMED, immaterial to trend

In [8]:

con.execute('''
SELECT COUNT(*) AS accounts_with_multiple_tz_labels
FROM (SELECT account_id FROM raw_calls GROUP BY 1 HAVING COUNT(DISTINCT timezone) > 1)
''').df()


,accounts_with_multiple_tz_labels
0,20874


### D. Vendor / disposition code changes — vendor: not confirmed; disposition synonym: confirmed

In [9]:

con.execute('''
SELECT vendor_id, COUNT(*) total_calls,
       ROUND(100.0*COUNT(*) FILTER (WHERE call_status='ANSWERED')/COUNT(*),1) answer_rate_pct
FROM raw_calls GROUP BY vendor_id ORDER BY answer_rate_pct DESC
''').df()


,vendor_id,total_calls,answer_rate_pct
0,VND0000014,6132,21.1
1,VND0000006,5989,20.9
2,VND0000010,6248,20.3
3,VND0000013,5974,20.1
4,VND0000011,6177,20.1
5,VND0000003,6068,19.9
6,VND0000008,5989,19.8
7,VND0000015,6150,19.8
8,VND0000002,6026,19.6
9,VND0000005,6153,19.5


In [10]:

con.execute('''
SELECT date_trunc('month', event_at) m, disposition_code, COUNT(*)
FROM raw_call_dispositions WHERE disposition_code IN ('PTP','PROMISE_TO_PAY') GROUP BY 1,2 ORDER BY 1,2
''').df()


,m,disposition_code,count_star()
0,2026-01-01,PROMISE_TO_PAY,588
1,2026-01-01,PTP,566
2,2026-02-01,PROMISE_TO_PAY,508
3,2026-02-01,PTP,483
4,2026-03-01,PROMISE_TO_PAY,557
5,2026-03-01,PTP,549
6,2026-04-01,PROMISE_TO_PAY,547
7,2026-04-01,PTP,538
8,2026-05-01,PROMISE_TO_PAY,543
9,2026-05-01,PTP,555


### E. Agent identity problems — CONFIRMED, severe

In [11]:

con.execute('''
SELECT COUNT(*) raw_rows, COUNT(DISTINCT agent_id) distinct_agent_id,
       COUNT(DISTINCT employee_code) distinct_employee_code,
       COUNT(DISTINCT agent_name) distinct_names
FROM raw_agents
''').df()


,raw_rows,distinct_agent_id,distinct_employee_code,distinct_names
0,30000,1000,1099,10


In [12]:

# one agent_id, radically different attributes on every row
con.execute('''
SELECT agent_id, employee_code, agent_name, vendor_id, team, status
FROM raw_agents WHERE agent_id = (SELECT agent_id FROM raw_agents GROUP BY 1 ORDER BY COUNT(*) DESC LIMIT 1)
LIMIT 8
''').df()


,agent_id,employee_code,agent_name,vendor_id,team,status
0,AGT0000367,EMP00210,Priya Mehta,VND0000012,T2,SUSPENDED
1,AGT0000367,EMP00644,Vikram Shah,VND0000015,T3,ACTIVE
2,AGT0000367,EMP00668,Rohan Patel,VND0000015,T2,ACTIVE
3,AGT0000367,EMP01066,Priya Mehta,VND0000008,T3,SUSPENDED
4,AGT0000367,EMP00034,Ananya Rao,VND0000012,DIGITAL,SUSPENDED
5,AGT0000367,EMP00909,Aarav Sharma,VND0000003,T2,ACTIVE
6,AGT0000367,EMP00770,Neha Singh,VND0000002,DIGITAL,ACTIVE
7,AGT0000367,EMP00695,Ananya Rao,VND0000003,T1,INACTIVE


### F. Portfolio mix changes — NOT CONFIRMED (stable mix rules out this driver)

In [13]:

con.execute('''
SELECT date_trunc('month', p.event_at) m, a.risk_segment, COUNT(DISTINCT p.account_id) n
FROM raw_payments p JOIN raw_accounts a USING(account_id)
WHERE p.payment_status='SUCCESS' GROUP BY 1,2
''').df().pivot(index='m', columns='risk_segment', values='n').apply(lambda r: 100*r/r.sum(), axis=1).round(1)


risk_segment,HIGH,LOW,MEDIUM,NPA
m,,,,
2026-01-01,25.6,24.8,25.7,24.0
2026-02-01,23.8,24.5,26.0,25.7
2026-03-01,24.4,25.8,24.0,25.8
2026-04-01,23.1,27.6,24.9,24.3
2026-05-01,26.6,25.3,24.8,23.3
2026-06-01,25.2,25.6,26.2,23.0
2026-07-01,24.8,25.7,25.0,24.5
2026-08-01,26.8,24.2,22.5,26.5


### G. Denominator manipulation — not confirmed in raw targeting data, but a live risk

In [14]:

con.execute("SELECT * FROM inv_priority_doseresponse").df()  # priority isn't the denominator issue, shown for completeness


,priority,n,n_paid,pay_rate_pct
0,1,4531,370,8.17
1,2,4539,335,7.38
2,3,4520,331,7.32
3,4,4358,321,7.37
4,5,4401,331,7.52
5,6,4432,342,7.72
6,7,4641,351,7.56
7,8,4587,375,8.18
8,9,4539,331,7.29
9,10,4452,339,7.61


In [15]:

con.execute('''
WITH d1 AS (SELECT COUNT(*) n FROM raw_accounts),
     d2 AS (SELECT COUNT(DISTINCT account_id) n FROM raw_calls WHERE event_at >= '2026-07-01' AND event_at < '2026-08-01'),
     d3 AS (SELECT COUNT(*) n FROM raw_accounts WHERE status='ACTIVE'),
     num AS (SELECT COUNT(DISTINCT account_id) n FROM raw_calls WHERE call_status='ANSWERED' AND event_at >= '2026-07-01' AND event_at < '2026-08-01')
SELECT (SELECT n FROM num) AS july_answered,
       ROUND(100.0*(SELECT n FROM num)/(SELECT n FROM d1),2) AS rate_vs_whole_portfolio,
       ROUND(100.0*(SELECT n FROM num)/(SELECT n FROM d2),2) AS rate_vs_worked_this_month,
       ROUND(100.0*(SELECT n FROM num)/(SELECT n FROM d3),2) AS rate_vs_active_only
'''
).df()


,july_answered,rate_vs_whole_portfolio,rate_vs_worked_this_month,rate_vs_active_only
0,2345,7.82,22.82,31.1


### H. Borrower identity problems — CONFIRMED, severe (found independently)
### I. `accounts.status` vs. event history — CONFIRMED, severe

In [16]:

print("Borrower identity conflicts:",
      con.execute("SELECT COUNT(*) FROM clean_borrowers WHERE identity_conflict_flag").fetchone()[0],
      "/", con.execute("SELECT COUNT(*) FROM clean_borrowers").fetchone()[0])
print("Status source conflicts:",
      con.execute("SELECT COUNT(*) FROM golden_accounts WHERE status_source_conflict_flag").fetchone()[0],
      "/", con.execute("SELECT COUNT(*) FROM golden_accounts WHERE status_from_history IS NOT NULL").fetchone()[0])


Borrower identity conflicts: 8518 / 11015
Status source conflicts: 22295 / 25999


## 4. Question 1 & 3 — Reconstructing performance and testing the "11% MoM" claim

Full narrative: `reports/performance_reconstruction.md`.

In [17]:

rec = con.execute("SELECT * FROM metric_monthly_recovery_mom").df()
rec


,event_month,recovery_rupees,n_success_payments,n_accounts_paid,recovery_per_account,prev_month_recovery,mom_pct_change
0,2026-01-01,1.872291e+08,2464,2374,78866.52,NaN,NaN
1,2026-02-01,1.701425e+08,2268,2173,78298.41,1.872291e+08,-9.13
2,2026-03-01,1.889124e+08,2524,2419,78095.24,1.701425e+08,11.03
3,2026-04-01,1.751380e+08,2406,2304,76014.78,1.889124e+08,-7.29
4,2026-05-01,1.842503e+08,2449,2344,78605.07,1.751380e+08,5.20
5,2026-06-01,1.755597e+08,2366,2286,76797.78,1.842503e+08,-4.72
6,2026-07-01,1.872423e+08,2441,2335,80189.41,1.755597e+08,6.65
7,2026-08-01,4.710970e+07,616,608,77483.05,1.872423e+08,-74.84


In [18]:

complete = rec[rec['event_month'] < pd.Timestamp('2026-08-01')].reset_index(drop=True)
vals = complete['recovery_rupees'].values
t = np.arange(len(vals))
slope, intercept, r, p, se = stats.linregress(t, vals)
print(f"Mean monthly recovery (Jan-Jul): Rs {vals.mean():,.0f}")
print(f"Std dev:                          Rs {vals.std(ddof=1):,.0f}  (CV {100*vals.std(ddof=1)/vals.mean():.1f}%)")
print(f"Linear trend: slope=Rs {slope:,.0f}/month, r^2={r**2:.4f}, p={p:.3f}")
print()
print(f"Feb->Mar change: {complete.loc[2,'mom_pct_change']:.2f}%  <-- the likely source of the '11% MoM' claim")
print(f"Mar vs Jan (skipping the Feb dip): {100*(complete.loc[2,'recovery_rupees']-complete.loc[0,'recovery_rupees'])/complete.loc[0,'recovery_rupees']:.2f}%")


Mean monthly recovery (Jan-Jul): Rs 181,210,610
Std dev:                          Rs 7,443,933  (CV 4.1%)
Linear trend: slope=Rs 221,852/month, r^2=0.0041, p=0.891

Feb->Mar change: 11.03%  <-- the likely source of the '11% MoM' claim
Mar vs Jan (skipping the Feb dip): 0.90%


In [19]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9,4.5))
months_complete = complete['event_month'].dt.strftime('%b')
ax.plot(months_complete, vals/1e7, marker='o', color='#2563eb', linewidth=2, label='Monthly recovery (Rs Cr)')
ax.axhline(vals.mean()/1e7, color='#94a3b8', linestyle='--', linewidth=1, label='7-month mean')
ax.fill_between(range(len(vals)), (vals.mean()-vals.std(ddof=1))/1e7, (vals.mean()+vals.std(ddof=1))/1e7,
                color='#94a3b8', alpha=0.15, label='+/-1 std dev band')
ax.set_ylabel('Recovery (Rs Crore)')
ax.set_title('Monthly recovery is flat within a noise band -- Feb->Mar "+11%" is a rebound off a dip, not a trend')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('recovery_trend.png', dpi=140)
plt.show()
print("Saved recovery_trend.png")


Saved recovery_trend.png


**Conclusion:** the reported 11% MoM improvement is a real arithmetic
fact about one month-pair (Feb→Mar) but not evidence of genuine, sustained
operational improvement — see §4 of `reports/performance_reconstruction.md`
for the full argument (flat 7-month trend, p=0.89; every month within ~1
std dev of the mean; March is only +0.9% vs. January once you don't start
from the trough).

## 5. Part 3 — Statistical investigation of drivers and named biases

Full writeup: `reports/statistical_investigation.md`.

In [20]:

df = con.execute('''
SELECT agent_id, COUNT(*) n_calls, AVG(CASE WHEN call_status='ANSWERED' THEN 1.0 ELSE 0 END) answer_rate
FROM raw_calls WHERE agent_id IS NOT NULL GROUP BY 1 HAVING COUNT(*)>=30
''').df()
p_hat = df['answer_rate'].mean()
n_avg = df['n_calls'].mean()
expected_noise_std = np.sqrt(p_hat*(1-p_hat)/n_avg)
print(f"Observed agent-to-agent answer-rate std: {df['answer_rate'].std():.4f}")
print(f"Expected std from sampling noise alone:  {expected_noise_std:.4f}")
print("--> observed ~= expected: agent identity shows no detectable skill effect beyond noise")


Observed agent-to-agent answer-rate std: 0.0428
Expected std from sampling noise alone:  0.0422
--> observed ~= expected: agent identity shows no detectable skill effect beyond noise


In [21]:

con.execute("SELECT * FROM inv_attempt_frequency_doseresponse").df()


,bucket,n_acct_months,n_paid,pay_rate_pct
0,1,70818,5313,7.50
1,2-3,21799,1646,7.55
2,4-6,546,36,6.59


In [22]:

con.execute("SELECT * FROM inv_whatsapp_doseresponse").df()


,bucket,n,n_paid,pay_rate_pct
0,1,46025,3473,7.55
1,2,6063,473,7.80
2,3-4,599,49,8.18
3,5+,2,0,0.00


## 5b. Multivariate confirmation — full statistical pipeline

Every check so far has been a simple bivariate cut (one factor at a time),
deliberately, per the assignment's preference for transparent methods over
complex ones. As a final, more rigorous confirmatory step, we build one
**account-month panel** (89,071 observations: every account x month with at
least one call attempt, Jan-Jul 2026) and fit a **multivariate logistic
regression** predicting whether the account paid that month, controlling for
month, risk segment, loan type, DPD bucket, attempt count, and whether the
account was touched by WhatsApp/SMS/Field that month — all simultaneously.
This directly tests whether the Feb-dip/Mar-rebound pattern survives once
mix and touch intensity are held constant (the formal version of the mix/
Simpson's-paradox check in Part 3), and whether any driver has an
independent effect once the others are controlled for.

In [23]:

panel = con.execute('''
WITH worked AS (
  SELECT DISTINCT account_id, date_trunc('month', event_at) m
  FROM raw_call_attempts WHERE event_at >= '2026-01-01' AND event_at < '2026-08-01'
),
attempts AS (
  SELECT account_id, date_trunc('month', event_at) m, COUNT(*) n_attempts
  FROM raw_call_attempts GROUP BY 1,2
),
wa AS (SELECT DISTINCT account_id, date_trunc('month', event_at) m FROM clean_whatsapp_events),
sms AS (SELECT DISTINCT account_id, date_trunc('month', event_at) m FROM raw_sms_events),
field AS (SELECT DISTINCT account_id, date_trunc('month', event_at) m FROM raw_field_visits),
paid AS (SELECT DISTINCT account_id, date_trunc('month', event_at) m FROM raw_payments WHERE payment_status='SUCCESS')
SELECT w.account_id, w.m AS month, a.risk_segment, a.loan_type, a.dpd_bucket,
       att.n_attempts,
       CASE WHEN wa.account_id IS NOT NULL THEN 1 ELSE 0 END AS touched_wa,
       CASE WHEN sms.account_id IS NOT NULL THEN 1 ELSE 0 END AS touched_sms,
       CASE WHEN field.account_id IS NOT NULL THEN 1 ELSE 0 END AS touched_field,
       CASE WHEN p.account_id IS NOT NULL THEN 1 ELSE 0 END AS paid
FROM worked w
JOIN clean_accounts a ON a.account_id = w.account_id
LEFT JOIN attempts att ON att.account_id=w.account_id AND att.m=w.m
LEFT JOIN wa ON wa.account_id=w.account_id AND wa.m=w.m
LEFT JOIN sms ON sms.account_id=w.account_id AND sms.m=w.m
LEFT JOIN field ON field.account_id=w.account_id AND field.m=w.m
LEFT JOIN paid p ON p.account_id=w.account_id AND p.m=w.m
''').df()
panel['month_str'] = panel['month'].dt.strftime('%Y-%m')
panel['n_attempts'] = panel['n_attempts'].fillna(0)
panel.to_csv('panel.csv', index=False)
print(panel.shape, "account-month observations; base payment rate:", round(panel['paid'].mean()*100,2), "%")
panel.head()


(89071, 11) account-month observations; base payment rate: 7.77 %


,account_id,month,risk_segment,loan_type,dpd_bucket,n_attempts,touched_wa,touched_sms,touched_field,paid,month_str
0,ACC0013603,2026-07-01,HIGH,CREDIT_CARD,90+,3,1,1,0,0,2026-07
1,ACC0011661,2026-05-01,HIGH,BNPL,1-30,3,1,1,0,0,2026-05
2,ACC0001252,2026-07-01,LOW,AUTO,1-30,2,1,1,0,0,2026-07
3,ACC0028165,2026-03-01,NPA,BNPL,61-90,1,1,1,0,0,2026-03
4,ACC0015691,2026-06-01,MEDIUM,AUTO,1-30,2,1,1,0,0,2026-06


In [24]:

import statsmodels.formula.api as smf
from scipy.stats import chi2

formula = ("paid ~ C(month_str, Treatment(reference='2026-01')) + C(risk_segment) "
           "+ C(loan_type) + C(dpd_bucket) + n_attempts + touched_wa + touched_sms + touched_field")
restricted_formula = "paid ~ C(risk_segment) + C(loan_type) + C(dpd_bucket) + n_attempts + touched_wa + touched_sms + touched_field"

# Naive fit (i.i.d. standard errors). The likelihood-ratio test below compares
# log-likelihoods, which does not depend on the SE assumption, so it is kept
# as a first pass -- but see the clustering correction in the next cell before
# treating any p-value here as final.
full_model = smf.logit(formula, data=panel).fit(disp=0)
restricted_model = smf.logit(restricted_formula, data=panel).fit(disp=0)

lr_stat = 2*(full_model.llf - restricted_model.llf)
df_diff = full_model.df_model - restricted_model.df_model
p_lr = chi2.sf(lr_stat, df_diff)
print(f"Overall model LLR p-value (vs. null, intercept-only): {full_model.llr_pvalue:.4f}")
print(f"LR test (naive/i.i.d.) -- month fixed effects jointly zero: LR={lr_stat:.2f}, df={df_diff:.0f}, p={p_lr:.4f}")


Overall model LLR p-value (vs. null, intercept-only): 0.5627
LR test (naive/i.i.d.) -- month fixed effects jointly zero: LR=4.14, df=6, p=0.6581


### Correcting for repeated observations per account

The panel has an average of **3.0 monthly rows per account** (29,360 unique
accounts across 89,071 account-months) — the same account can appear in up
to 7 rows. Ordinary logistic-regression standard errors assume every row is
an independent draw, which is false here: an account's unobserved propensity
to pay is correlated across its own months. Left uncorrected this can distort
inference in either direction, so before treating the result above as final
it is re-estimated with **cluster-robust standard errors, clustered by
`account_id`**, and the joint hypothesis is re-tested with the correct tool
for a clustered design — a cluster-robust Wald test — rather than the
likelihood-ratio test, which assumes i.i.d. observations.

In [25]:

full_model_cl = smf.logit(formula, data=panel).fit(
    disp=0, cov_type='cluster', cov_kwds={'groups': panel['account_id']}
)

# Cluster-robust Wald test for "all 6 month effects = 0", using the
# cluster-robust covariance matrix (the correct joint test under clustering).
month_terms = [p for p in full_model_cl.params.index if p.startswith('C(month_str')]
r_matrix = np.zeros((len(month_terms), len(full_model_cl.params)))
for i, term in enumerate(month_terms):
    r_matrix[i, full_model_cl.params.index.get_loc(term)] = 1
wald = full_model_cl.wald_test(r_matrix, scalar=True)
n_clusters = panel['account_id'].nunique()
print(f"Cluster-robust Wald test (clustered by account_id, {n_clusters:,} clusters):")
print(f"chi2={float(wald.statistic):.3f}, df={int(wald.df_denom)}, p={float(wald.pvalue):.4f}")
print()

ci = full_model_cl.conf_int()
month_labels = ['Feb','Mar','Apr','May','Jun','Jul']
or_table = pd.DataFrame({
    'month': month_labels,
    'odds_ratio_vs_Jan': np.exp(full_model_cl.params[month_terms].values),
    'ci_low': np.exp(ci.loc[month_terms, 0].values),
    'ci_high': np.exp(ci.loc[month_terms, 1].values),
    'p_cluster_robust': full_model_cl.pvalues[month_terms].values,
})
print("Month effects on odds of payment (ref = Jan), cluster-robust 95% CI:")
print(or_table.round(4).to_string(index=False))


Cluster-robust Wald test (clustered by account_id, 29,360 clusters):
chi2=4.131, df=6, p=0.6589

Month effects on odds of payment (ref = Jan), cluster-robust 95% CI:
month  odds_ratio_vs_Jan  ci_low  ci_high  p_cluster_robust
  Feb             0.9167  0.8354   1.0060            0.0666
  Mar             0.9701  0.8864   1.0617            0.5092
  Apr             0.9421  0.8603   1.0317            0.1980
  May             0.9477  0.8661   1.0369            0.2416
  Jun             0.9491  0.8667   1.0393            0.2597
  Jul             0.9392  0.8580   1.0280            0.1736


**[Fact]** Clustering barely moves the result (Wald p = 0.659 vs. the
naive LR p = 0.658) — this data has low within-account correlation across
months, so the naive test was not misleading here, but checking rather than
assuming was necessary to say that with confidence. Every month's 95%
confidence interval for the odds ratio comfortably spans 1.00 (the tightest,
February, still runs 0.84–1.01), so the data rules out not just "no effect"
but any effect large enough to matter at the odds-ratio scale, in every
single month.

### Is this a conclusive null, or just an underpowered test?

Failing to reject a null hypothesis is only informative if the test had a
real chance to reject it. With 89,071 account-months, this design should
have plenty of power — but "should" is a claim to check, not assume, so the
minimum detectable effect (MDE) is computed directly: the smallest true
swing in the monthly payment rate this design would catch 80% of the time at
α = 0.05, using a two-proportion power calculation against the ~12,724
account-months observed per calendar month.

In [26]:

from statsmodels.stats.power import NormalIndPower

base_rate = panel['paid'].mean()
n_per_month = panel.groupby('month_str').size().mean()

power_calc = NormalIndPower()
mde_h = power_calc.solve_power(effect_size=None, nobs1=n_per_month, alpha=0.05,
                                power=0.80, ratio=1, alternative='two-sided')
# invert Cohen's h back to an absolute proportion difference around base_rate
phi1 = 2*np.arcsin(np.sqrt(base_rate))
p2_hi = np.sin((phi1 + mde_h)/2)**2
p2_lo = np.sin((phi1 - mde_h)/2)**2
mde_abs_pp = ((p2_hi - base_rate) + (base_rate - p2_lo)) / 2 * 100
mde_rel_pct = mde_abs_pp / (base_rate*100) * 100

largest_month_effect_pp = (np.exp(full_model_cl.params[month_terms]).sub(1).abs().max()) * base_rate * 100

print(f"Base monthly payment rate: {base_rate*100:.2f}%")
print(f"Account-months per calendar month (avg): {n_per_month:,.0f}")
print(f"Minimum detectable effect, 80% power / alpha=0.05: +/-{mde_abs_pp:.2f}pp ({mde_rel_pct:.1f}% relative)")
print(f"Largest of the 6 observed month effects, translated to pp at the base rate: {largest_month_effect_pp:.2f}pp")
print(f"--> the largest effect actually observed is well inside the band this design could detect,")
print(f"    so the null is a conclusive 'not there', not an artifact of insufficient sample size.")


Base monthly payment rate: 7.77%
Account-months per calendar month (avg): 12,724
Minimum detectable effect, 80% power / alpha=0.05: +/-0.94pp (12.1% relative)
Largest of the 6 observed month effects, translated to pp at the base rate: 0.65pp
--> the largest effect actually observed is well inside the band this design could detect,
    so the null is a conclusive 'not there', not an artifact of insufficient sample size.


**[Fact]** At 80% power this design would have caught a true swing of
roughly **±0.9 percentage points (≈12% relative)** in the underlying monthly
payment rate — the same order of magnitude as the swing that would be needed
to produce a genuine ~11% change in aggregate recovery. The largest month
effect actually estimated (February, the closest to significance) is
**~0.65pp**, smaller than the detectable floor. **This is the second
strongest piece of evidence in the analysis**, alongside the cluster-robust
regression itself: the test was well-powered to find a Feb→Mar-sized effect
if one existed, and it did not find one.

**This is the strongest evidence in the whole analysis that the reported
trend is not real.** Even in a full multivariate model that gives the "Feb
dip / Mar rebound" story every chance to survive — controlling for exactly
the mix and intensity variables that could produce a spurious pattern, with
standard errors corrected for repeated observations per account, and checked
against a power analysis to rule out "the test just couldn't have found it"
— the month fixed effects are jointly indistinguishable from zero
(cluster-robust Wald p = 0.659; naive LR p = 0.658), and the overall model
does not explain the outcome any better than a flat intercept (LLR p =
0.563). None of risk segment, loan type, DPD bucket, attempt count, or
channel-touch indicators reach significance either. The data is consistent
with **one stable base payment rate (~7.8% of worked accounts per month)
plus noise** — nothing more structured than that.

## 6. Part 4 — Counterfactual methodology

Full design + caveats: `reports/counterfactual_methodology.md`. No real,
time-ordered strategy cutover exists in this extract (`strategy_version`
values are concurrent throughout Jan-May), so the illustrative test below is
a cross-sectional proxy, not a true DiD.

In [27]:

sv = con.execute('''
WITH touches AS (SELECT DISTINCT campaign_id, account_id, event_at FROM golden_calls),
conv AS (
  SELECT t.campaign_id, t.account_id,
         MAX(CASE WHEN p.event_at BETWEEN t.event_at AND t.event_at + INTERVAL 14 DAY THEN 1 ELSE 0 END) AS converted
  FROM touches t LEFT JOIN golden_payments p ON p.account_id = t.account_id AND p.payment_status='SUCCESS'
  GROUP BY 1,2
)
SELECT camp.strategy_version, COUNT(DISTINCT conv.account_id) n_accounts, SUM(conv.converted) n_converted
FROM conv JOIN clean_campaigns camp USING(campaign_id)
GROUP BY 1
''').df()
sv['group'] = sv['strategy_version'].map({'legacy':'OLD','v1':'OLD','v2':'NEW','v3':'NEW'})
g = sv.groupby('group')[['n_accounts','n_converted']].sum()
g['conv_rate_pct'] = 100*g['n_converted']/g['n_accounts']
print(g)

count = g['n_converted'].values
nobs = g['n_accounts'].values
p_pool = count.sum()/nobs.sum()
se = np.sqrt(p_pool*(1-p_pool)*(1/nobs[0]+1/nobs[1]))
z = (count[0]/nobs[0]-count[1]/nobs[1])/se
from scipy.stats import norm
pval = 2*(1-norm.cdf(abs(z)))
print(f"\nz={z:.3f}, p={pval:.3f} -- no statistically detectable difference between OLD and NEW strategy groups")


       n_accounts  n_converted  conv_rate_pct
group                                        
NEW         31292       1540.0       4.921386
OLD         31486       1607.0       5.103856

z=-1.048, p=0.295 -- no statistically detectable difference between OLD and NEW strategy groups


## 7. Question 4 — Where should the ₹10 Cr go

Full recommendation with financial model: `reports/investment_recommendation.md`.
Key supporting numbers reproduced here:

In [28]:

con.execute("SELECT * FROM inv_capacity_anchors").df()


,avg_attempts_per_month,avg_agent_hours_per_month,attempts_per_agent_hour,avg_monthly_recovery,avg_recovery_per_agent_hour
0,16518.142857,10883.680952,1.52,1.812106e+08,16644.691429


In [29]:

con.execute("SELECT * FROM inv_whatsapp_reach").df()


,accounts_ever_touched,total_accounts,accounts_never_touched,avg_events_per_touched_account
0,25924,30000,4076,2.314458


**Recommendation: WhatsApp / digital engagement, deployed as a gated
two-phase investment** (₹1.2 Cr pilot RCT, then ₹8.8 Cr scaled rollout
conditional on the pilot showing a statistically significant lift) — the
only lever with even a directional (not yet significant) positive
dose-response in this data, and by far the cheapest to test and scale. See
`reports/investment_recommendation.md` §1-4 for the full reasoning, why the
other five options are not recommended, and the ROI/break-even table under
downside / base / upside incrementality assumptions.

## 8. Conclusion

1. **What happened:** essentially nothing, operationally — recovery has been
   flat (p=0.89) for the 7 complete months in this extract, within a normal
   ~4% month-to-month noise band.
2. **Why:** no tested driver (agent, tenure, vendor, channel, campaign,
   attempt frequency, targeting-strategy proxy, targeting priority) shows an
   effect distinguishable from sampling noise. Geography and language could
   not be tested at all (data gaps).
3. **Is the 11% real:** no — it's real arithmetic, cherry-picked from a
   rebound off an unusually weak February, not a sustained trend.
4. **Where to invest ₹10 Cr:** WhatsApp/digital cadence, gated by a ₹1.2 Cr
   pilot RCT before the remaining ₹8.8 Cr is committed — because it is the
   only lever with a (weak, unproven) positive signal and the cheapest to
   test properly before betting the rest of the budget.
